# Metrics Extension - BIOMQM Evaluation

This notebook runs three additional evaluation metrics:
1. **BioDeBERTa**: Biomedical semantic similarity using `NeuML/pubmedbert-base-embeddings`
2. **NLI Classifier**: Natural Language Inference using `facebook/bart-large-mnli`
3. **LLM Judge**: Qwen as NLI judge for comparison with classifier

## 0. Environment Detection

In [ ]:
import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
elif IN_KAGGLE:
    print('Running on Kaggle - models will be cached in /root/.cache')
else:
    print('Running locally')

## 1. Install Dependencies & Clone Repository

In [ ]:
import subprocess

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 
                'transformers', 'torch', 'accelerate', 'sentencepiece'], check=True)
print('Dependencies installed!')

# Clone repository
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        print(f'Cloning repository to {PROJECT_ROOT}...')
        subprocess.run(['git', 'clone', 
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', 
                        PROJECT_ROOT], check=True)
        print('Clone complete!')
    else:
        print(f'Repository already exists at {PROJECT_ROOT}')
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone', 
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', 
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

## 2. Pre-download Models

Download all models needed for the metrics extension.

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForCausalLM
import torch

print('=== Downloading/Loading Models ===')
print('This may take a while on first run...\n')

MODELS = {
    'biodeberta': 'NeuML/pubmedbert-base-embeddings',
    'nli': 'facebook/bart-large-mnli',
    'qwen': 'Qwen/Qwen2.5-3B-Instruct'
}

# Download BioDeBERTa
print(f"[1/3] Loading {MODELS['biodeberta']}...")
tokenizer = AutoTokenizer.from_pretrained(MODELS['biodeberta'])
model = AutoModel.from_pretrained(MODELS['biodeberta'])
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ BioDeBERTa cached')

# Download NLI model
print(f"[2/3] Loading {MODELS['nli']}...")
tokenizer = AutoTokenizer.from_pretrained(MODELS['nli'])
model = AutoModelForSequenceClassification.from_pretrained(MODELS['nli'])
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ NLI classifier cached')

# Download Qwen
print(f"[3/3] Loading {MODELS['qwen']}...")
tokenizer = AutoTokenizer.from_pretrained(MODELS['qwen'])
model = AutoModelForCausalLM.from_pretrained(MODELS['qwen'], torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ Qwen cached')

print('\n=== All models cached! ===')

## 3. Setup Paths

In [ ]:
# Verify input file exists
MAPPED_FILE = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/mapping.jsonl"

if os.path.exists(MAPPED_FILE):
    with open(MAPPED_FILE, 'r') as f:
        n_lines = sum(1 for _ in f)
    print(f"\n✓ Mapped file found ({n_lines} rows)")
else:
    print(f"\n✗ ERROR: Mapped file NOT found: {MAPPED_FILE}")
    print("Run the baseline pipeline first!")

## 4. BioDeBERTa Semantic Similarity

Calculates cosine similarity using biomedical embeddings.

In [ ]:
# BioDeBERTa Evaluation
script_path = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension/evaluation/biodeberta/biodeberta.py"
mapped_file_path = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/mapping.jsonl"
output_base_dir = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension"

cmd = [
    sys.executable, "-u",
    script_path,
    "--mapped_file_path", mapped_file_path,
    "--output_base_dir", output_base_dir
]

print(f"Running BioDeBERTa Evaluation...")
print(f"Script: {script_path}")
print(f"Input: {mapped_file_path}")
print(f"Output: {output_base_dir}")

subprocess.run(cmd, check=True)
print("✓ BioDeBERTa complete!")

## 5. NLI Classifier

Classifies entailment/neutral/contradiction using DeBERTa-v3-large-mnli.

In [ ]:
# NLI Classifier Evaluation
script_path = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension/evaluation/nli/nli_classifier.py"
mapped_file_path = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/mapping.jsonl"
output_base_dir = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension"

cmd = [
    sys.executable, "-u",
    script_path,
    "--mapped_file_path", mapped_file_path,
    "--output_base_dir", output_base_dir
]

print(f"Running NLI Evaluation...")
print(f"Script: {script_path}")
print(f"Input: {mapped_file_path}")
print(f"Output: {output_base_dir}")

subprocess.run(cmd, check=True)
print("✓ NLI Classifier complete!")

## 6. LLM Judge

Uses Qwen as judge for NLI classification. Compare with DeBERTa classifier for agreement analysis.

In [ ]:
# LLM Judge Evaluation
script_path = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension/evaluation/llm-judge/llm_judge.py"
mapped_file_path = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/mapping.jsonl"
output_base_dir = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension"

cmd = [
    sys.executable, "-u",
    script_path,
    "--mapped_file_path", mapped_file_path,
    "--output_base_dir", output_base_dir
]

print(f"Running LLM Judge Evaluation...")
print(f"Script: {script_path}")
print(f"Input: {mapped_file_path}")
print(f"Output: {output_base_dir}")

subprocess.run(cmd, check=True)
print("✓ LLM Judge complete!")

## Summary

The Metrics Extension pipeline is complete. Check the output files:

- **BioDeBERTa**: `{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension/results/biodeberta/`
- **NLI Classifier**: `{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension/results/nli/`
- **LLM Judge**: `{PROJECT_ROOT}/results Qwen3B baseline/biomqm/metrics-extension/results/llm-judge/`